# Cubic Splines and Boundary Conditions

Runge's phenomenon shows that a single high-degree polynomial through many points can diverge. A spline gives up the single polynomial and uses low-degree pieces instead, joined with whatever smoothness you ask for. This notebook builds the piecewise-linear and cubic cases, solves the tridiagonal system behind the cubic, and looks at what the choice of boundary condition does to the fit.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## The tridiagonal system

Let $M_i = s''(x_i)$ denote the second derivatives at the nodes and $h_i = x_{i+1}-x_i$. Continuity of the first derivative at each interior node gives, for $i = 1,\dots,n-1$,

$$ h_{i-1} M_{i-1} + 2(h_{i-1}+h_i) M_i + h_i M_{i+1} = 6\left(\frac{y_{i+1}-y_i}{h_i} - \frac{y_i-y_{i-1}}{h_{i-1}}\right). $$

This is tridiagonal in the $M_i$ and solves in $\mathcal{O}(n)$. It supplies $n-1$ equations for $n+1$ unknowns, and the two missing equations are the boundary conditions: natural, meaning $M_0=M_n=0$, or clamped, with the end slopes prescribed.

In [ ]:
# ---------------------------------------------------------------------------
# Piecewise-linear interpolation
# ---------------------------------------------------------------------------
# The simplest spline connects the dots with straight segments. It is continuous
# but has kinks, since its first derivative jumps at the nodes. Cubic splines below
# remove those kinks by matching first and second derivatives too.

def piecewise_linear(x_nodes, y_nodes, x):
    """Evaluate the connect-the-dots interpolant, which is what np.interp does."""
    return np.interp(x, x_nodes, y_nodes)

In [ ]:
# ---------------------------------------------------------------------------
# Cubic spline, step 1: the interior equations
# ---------------------------------------------------------------------------
# A cubic spline is a cubic on each interval [x_i, x_{i+1}], stitched so that
# the function, its first derivative, and its second derivative are continuous
# at the interior nodes. The unknowns are the moments M_i = s''(x_i), and
# first-derivative continuity gives one equation per interior node.

def spline_interior_equations(x, y):
    """Assemble the tridiagonal system A M = d for the spline moments.

    Only the interior rows 1..n-1 are filled; rows 0 and n are left zero
    for the boundary condition (next cell) to supply.

    Parameters
    ----------
    x, y : arrays (n+1,)  sorted nodes and values

    Returns
    -------
    A : array (n+1, n+1)
    d : array (n+1,)
    """
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    n = len(x) - 1
    h = np.diff(x)
    A = np.zeros((n + 1, n + 1))
    d = np.zeros(n + 1)
    for i in range(1, n):
        A[i, i - 1] = h[i - 1]
        A[i, i]     = 2 * (h[i - 1] + h[i])
        A[i, i + 1] = h[i]
        d[i] = 6 * ((y[i + 1] - y[i]) / h[i] - (y[i] - y[i - 1]) / h[i - 1])
    return A, d

In [ ]:
# ---------------------------------------------------------------------------
# Cubic spline, step 2: boundary conditions, and the solve
# ---------------------------------------------------------------------------
# The interior gives n-1 equations for n+1 unknowns. The last two rows are
# the boundary condition:
#   "natural"  : M_0 = M_n = 0             (zero curvature at the ends)
#   "clamped"  : prescribe s'(x_0), s'(x_n)  (default: the end secant slopes)

def apply_boundary(A, d, x, y, bc, slopes=None):
    """Fill rows 0 and n of the system with the chosen boundary condition."""
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    n = len(x) - 1
    h = np.diff(x)
    if bc == "natural":
        A[0, 0] = 1.0; d[0] = 0.0
        A[n, n] = 1.0; d[n] = 0.0
    elif bc == "clamped":
        if slopes is None:
            slopes = ((y[1] - y[0]) / h[0], (y[n] - y[n - 1]) / h[n - 1])
        fp0, fpn = slopes
        A[0, 0] = 2 * h[0]; A[0, 1] = h[0]
        d[0] = 6 * ((y[1] - y[0]) / h[0] - fp0)
        A[n, n - 1] = h[n - 1]; A[n, n] = 2 * h[n - 1]
        d[n] = 6 * (fpn - (y[n] - y[n - 1]) / h[n - 1])
    else:
        raise ValueError("bc must be 'natural' or 'clamped'")


def cubic_spline_moments(x, y, bc="natural", slopes=None):
    """Second-derivative moments M_i of the cubic spline through (x, y)."""
    A, d = spline_interior_equations(x, y)
    apply_boundary(A, d, x, y, bc, slopes)
    return np.linalg.solve(A, d)

In [ ]:
# ---------------------------------------------------------------------------
# The system for a small data set
# ---------------------------------------------------------------------------
x_demo = np.array([0.0, 1.0, 3.0, 6.0])
y_demo = np.array([1.0, 4.0, 2.0, 8.0])

A, d = spline_interior_equations(x_demo, y_demo)
apply_boundary(A, d, x_demo, y_demo, "natural")
print("A =")
print(A)
print("d =", d)
print("M =", np.round(np.linalg.solve(A, d), 4))

In [ ]:
# ---------------------------------------------------------------------------
# Cubic spline, step 3: evaluate
# ---------------------------------------------------------------------------
def cubic_spline_eval(x, y, M, xq):
    """Evaluate the cubic spline (given moments M) at query points xq.

    On [x_i, x_{i+1}] the spline is the moment formula combining M_i,
    M_{i+1} and the two endpoint values. A query outside the data is
    served by the nearest panel.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    xq = np.asarray(xq, float)
    out = np.empty_like(xq)
    for k, t in enumerate(xq):
        i = 0
        while x[i + 1] < t and i < len(x) - 2:
            i = i + 1
        a = x[i + 1] - t
        b = t - x[i]
        h = x[i + 1] - x[i]
        out[k] = (M[i] * a**3 + M[i + 1] * b**3) / (6 * h) \
                 + (y[i] / h - M[i] * h / 6) * a \
                 + (y[i + 1] / h - M[i + 1] * h / 6) * b
    return out

In [ ]:
# ---------------------------------------------------------------------------
# Compare linear vs cubic, natural vs clamped
# ---------------------------------------------------------------------------
def f_demo(x):
    """Target function chosen so the boundary condition affects the ends."""
    return np.cos(x) * np.exp(-0.15 * x)

def show_spline(n_nodes=7, spline="cubic", bc="natural", a=0.0, b=10.0):
    """Sample f at n_nodes points and plot the chosen interpolant against f."""
    x = np.linspace(a, b, n_nodes)
    y = f_demo(x)
    xx = np.linspace(a, b, 600)

    plt.figure()
    plt.plot(xx, f_demo(xx), "k-", lw=1.5, alpha=0.6, label="f(x)")
    if spline == "linear":
        plt.plot(xx, piecewise_linear(x, y, xx), "g-", lw=2,
                 label="piecewise linear")
    else:
        M = cubic_spline_moments(x, y, bc=bc)
        plt.plot(xx, cubic_spline_eval(x, y, M, xx), "r-", lw=2,
                 label=f"cubic spline ({bc})")
    plt.plot(x, y, "bo", ms=6, label="nodes")
    plt.legend(); plt.xlabel("x")
    plt.title(f"{spline} interpolation" + (f", {bc} BC" if spline == "cubic" else ""))
    plt.show()

show_spline(7, "cubic", "natural")

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: node count, linear vs cubic, and the boundary condition
# ---------------------------------------------------------------------------
# With few nodes, switch bc between "natural" and "clamped" and watch how the
# curve bends differently near the two ends. The interior is barely affected.
interact(
    show_spline,
    n_nodes=IntSlider(min=3, max=15, step=1, value=7, description="# nodes"),
    spline=Dropdown(options=["cubic", "linear"], value="cubic", description="type"),
    bc=Dropdown(options=["natural", "clamped"], value="natural", description="cubic BC"),
    a=(-2.0, 2.0, 1.0), b=(6.0, 14.0, 1.0),
);

## The cost of the boundary condition

The natural condition sets $s''=0$ at the two ends. That is usually false for the function being interpolated, so the fit suffers there. The question worth asking is how far inward the damage reaches. The cell below measures the largest error on the end panels, on everything except the end panels, and on the middle third, for three fits: natural, clamped with the exact end slopes, and the linear spline. It then refines the mesh and reports the factor by which each error falls at every doubling.

In [ ]:
def df_demo(x):
    """Exact derivative of f_demo, so the clamped condition can be exact."""
    return np.exp(-0.15 * x) * (-np.sin(x) - 0.15 * np.cos(x))

def error_split(m):
    """Max error on the end panels, outside them, and over the middle third."""
    x = np.linspace(0.0, 10.0, m)
    y = f_demo(x)
    fine = np.linspace(0.0, 10.0, 4001)
    left, right = fine <= x[1], fine >= x[-2]
    inner = ~(left | right)                       # everything but the end panels
    middle = (fine > 10.0 / 3) & (fine < 20.0 / 3)  # a fixed fraction of [0, 10]

    def regions(e):
        return (e[left].max(), e[right].max(), e[inner].max(), e[middle].max())

    out = {}
    for name in ["natural", "clamped"]:
        M = cubic_spline_moments(x, y, bc=name,
                                 slopes=(df_demo(x[0]), df_demo(x[-1])))
        out[name] = regions(np.abs(f_demo(fine) - cubic_spline_eval(x, y, M, fine)))
    out["linear"] = regions(np.abs(f_demo(fine) - piecewise_linear(x, y, fine)))
    return out

print("m = 9 nodes")
for name, (lo, hi, inner, mid) in error_split(9).items():
    print(f"  {name:8s} first panel {lo:.3e}   last panel {hi:.3e}   "
          f"outside the end panels {inner:.3e}   middle third {mid:.3e}")

nan = float("nan")
for name in ["natural", "clamped"]:
    print(f"\n{name} condition, error and the factor it falls by at each doubling")
    print(f"{'m':>5} {'end panels':>19} {'outside them':>19} {'middle third':>19}")
    prev = None
    for m in [9, 17, 33, 65, 129]:
        r = error_split(m)
        end, out, mid = max(r[name][0], r[name][1]), r[name][2], r[name][3]
        f1, f2, f3 = (prev[0] / end, prev[1] / out, prev[2] / mid) if prev else (nan, nan, nan)
        print(f"{m:5d} {end:12.2e} (x{f1:4.1f}) {out:12.2e} (x{f2:4.1f})"
              f" {mid:12.2e} (x{f3:4.1f})")
        prev = (end, out, mid)

The natural error falls by about $4$ per doubling and the clamped error by about $16$, so the rates are $\mathcal{O}(h^2)$ and $\mathcal{O}(h^4)$. Those rates hold on the end panels and on everything outside them too, which says the natural condition costs two orders globally rather than only near the ends. Over the middle third, though, the two conditions agree to every digit printed and both run at $\mathcal{O}(h^4)$.

Both readings are correct because the boundary error decays by a fixed factor per panel. A region defined as a fixed number of panels from the end stays polluted however fine the mesh gets, while a region defined as a fraction of the interval sits ever more panels away from the end as the mesh refines. For this $f$, $|f''(0)| \approx 1$ against $|f''(10)| \approx 0.15$, so the left end absorbs more of the error, and the first two columns show it. Clamp when you know the end derivatives. When you do not, clamping to the end secant slope is a guess, and it can come out worse than natural.

## Against a single polynomial

Splines exist because of Runge's phenomenon, so here is the head-to-head. Both interpolants go through the same 21 equally spaced samples.

In [ ]:
def runge(x):
    """Runge's function 1/(1 + 25 x^2) on [-1, 1]."""
    return 1.0 / (1.0 + 25.0 * x**2)

def lagrange_eval(nodes, values, x):
    """Evaluate the interpolating polynomial in Lagrange form."""
    nodes = np.asarray(nodes, dtype=float); values = np.asarray(values, dtype=float)
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    for i in range(len(nodes)):
        Li = np.ones_like(x)
        for j in range(len(nodes)):
            if j != i:
                Li *= (x - nodes[j]) / (nodes[i] - nodes[j])
        result += values[i] * Li
    return result

xr = np.linspace(-1.0, 1.0, 21)
yr = runge(xr)
fine = np.linspace(-1.0, 1.0, 2001)
poly = lagrange_eval(xr, yr, fine)
spl = cubic_spline_eval(xr, yr, cubic_spline_moments(xr, yr, bc="natural"), fine)

print(f"degree 20 polynomial   max error {np.abs(runge(fine) - poly).max():.3e}")
print(f"natural cubic spline   max error {np.abs(runge(fine) - spl).max():.3e}")

plt.figure()
plt.plot(fine, runge(fine), "k-", lw=1.5, label="f")
plt.plot(fine, poly, "r--", lw=1.5, label="degree 20 polynomial")
plt.plot(fine, spl, "b-", lw=1.5, label="natural cubic spline")
plt.plot(xr, yr, "ko", ms=4)
plt.ylim(-1.0, 2.0); plt.xlabel("x"); plt.legend()
plt.title("21 equally spaced nodes"); plt.show()

More nodes means a higher-degree polynomial, and with it a larger $\max_x|\omega(x)|$ across the whole interval, which is what drives the equispaced divergence. A cubic spline stays cubic however many nodes you give it, and each data value affects only the panels beside it. Refining the mesh therefore shrinks the error rather than pushing it into endpoint oscillations.

## Summary

A cubic spline matches value, first derivative, and second derivative at every interior node, so it has neither the kinks of the linear spline nor the endpoint divergence of one high-degree polynomial. The linear spline stays continuous, but its first derivative jumps at every node, and the kinks are where you see that. The moments $M_i$ come from a tridiagonal system, and the boundary condition supplies the last two degrees of freedom. That choice costs more than it looks: natural gives $\mathcal{O}(h^2)$, clamping with the exact end slopes gives $\mathcal{O}(h^4)$, and the gap survives outside the end panels even though it disappears over the middle of the interval.

## Things to try

- With only a few nodes, switch `bc` between natural and clamped. The two ends bend differently while the middle barely moves.
- Switch to the linear spline and find the kinks at the nodes. That is $s'$ jumping.
- Replace `f_demo` with a function whose second derivative is large at the endpoints, then see what the natural condition costs you there.